# torchX

A comprehensive guide to torchX for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

TorchX is a toolkit from the PyTorch ecosystem for **authoring and launching PyTorch applications** on a variety of schedulers (local, Kubernetes, Slurm, Ray, etc.). It focuses on describing distributed jobs as reusable components and running them on different backends.

### What is it?

At a high level, **TorchX**:

- Lets you define **components** (Python entrypoints) that represent training or inference jobs.  
- Provides a `torchx run` CLI and Python APIs to launch those components on many schedulers.  
- Helps you standardize how you run distributed PyTorch jobs across environments.

### Why use it?

Key benefits of using TorchX:

- **Environment-agnostic launching**: Same component can run locally, on Kubernetes, Slurm, Ray, etc.  
- **Composition**: Combine multiple components into pipelines or multi-stage apps.  
- **Separation of concerns**: Training code is decoupled from how/where it is launched.

### When to use it?

TorchX is particularly useful when:

- You run **PyTorch training/inference** in multiple environments and want a consistent interface.  
- You want to **encapsulate distributed job specs** (world size, resources, image, entrypoint) in reusable components.  
- You need to support different schedulers without rewriting your training scripts.

## Key Features

### Core Capabilities of TorchX

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Components** | Python-callable descriptions of jobs (e.g., distributed trainers). | Reusable building blocks for training and inference jobs. |
| **Schedulers** | Pluggable backends: local, Kubernetes, Slurm, Ray, etc. | Run the same job in different environments. |
| **`torchx run` CLI** | Command-line entrypoint for launching components. | Easy to script and integrate into CI/CD. |
| **Pipelines / multi-stage apps** | Compose components into end-to-end workflows. | Express complex ML pipelines (data prep, training, evaluation). |
| **Logging & tracking hooks** | Integrate with experiment tracking and logging systems. | Keep runs observable across environments. |

## Architecture Overview

TorchX separates **what** to run (components) from **where/how** to run it (schedulers).

```text
+---------------------------+
|    TorchX Component       |
| (Python function/module)  |
+--------------+------------+
               |
               |  torchx run -s <scheduler>
               v
+---------------------------+
|   TorchX Runner +        |
|   Scheduler Plugin        |
+--------------+------------+
               |
               |  translates to Job spec
               v
+---------------------------+
|  Target Scheduler         |
|  (local, k8s, Slurm, ...) |
+---------------------------+
```

### Key components

1. **Component definition**  
   - A Python function or entrypoint that TorchX can call to construct a job spec.

2. **Runner & schedulers**  
   - Runner interprets component arguments and hands off to a scheduler implementation (local, k8s, etc.).

3. **Underlying cluster**  
   - Actual resources (GPUs, CPUs) provisioned by Kubernetes, Slurm, Ray, etc., where the job runs.

## Installation

### Prerequisites

- Python 3.8+.
- PyTorch installed for your environment.
- Optional: Docker, Kubernetes, Slurm, or other schedulers where you plan to run jobs.

### Install TorchX

```bash
pip install torchx
```

Refer to the TorchX docs for scheduler-specific setup (e.g., configuring Kubernetes or Slurm plugins).

In [ ]:
# Quick install helper for notebooks (uncomment to run)
# !pip install torchx torch torchvision

## Basic Usage

### Example: running a built-in DDP trainer locally

TorchX includes built-in components for common patterns, such as distributed data-parallel (DDP) training.

You can launch a simple DDP example (conceptually) with:

```bash
torchx run -s local_cwd dist.ddp \
  --script train.py \
  --nnodes 1 \
  --nproc_per_node 2
```

This tells TorchX to:

- Use the **local_cwd** scheduler (runs on your local machine using the current working directory).  
- Run the `dist.ddp` component with your `train.py` script.  
- Use 2 processes (typically 2 GPUs) on 1 node.

In [ ]:
# Sketch: example TorchX component in Python (conceptual)

from torchx.components.dist import ddp

# Example: programmatic launch (not executed here)
# from torchx.runner import get_runner
#
# runner = get_runner("local_cwd")
# app = ddp(
#     script="train.py",  # your training script
#     nnodes=1,
#     nproc_per_node=2,
# )
#
# app_handle = runner.run(app, scheduler="local_cwd")
# print("Launched app:", app_handle)

print("TorchX can run built-in components like dist.ddp via CLI or Python APIs.")

## Advanced Features

- **Custom components**: Define your own Python components that build more complex apps (e.g., multi-stage training pipelines).  
- **Multiple schedulers**: Swap schedulers (local, k8s, Slurm, Ray) without changing component logic.  
- **Job inspection and logs**: Use TorchX commands or underlying schedulers to inspect job status and logs.  
- **Integration with PyTorch Ecosystem**: Works well with PyTorch DDP, FSDP, and other distributed tools inside your `train.py`.

In [ ]:
# Placeholder for defining a richer custom TorchX component

print("See TorchX docs for full custom component examples (e.g., Lightning trainers).")

## Use Cases

- Running **distributed PyTorch training** on Kubernetes, Slurm, or local environments.  
- Building **portable job definitions** for CI/CD and experimentation.  
- Composing multi-step applications (data prep, training, evaluation, interpretation) from reusable components.

## Best Practices

1. **Keep training scripts scheduler-agnostic**: Use standard PyTorch distributed patterns inside `train.py`; let TorchX handle launching.  
2. **Version your components**: Treat component definitions like code artifacts; changes affect how jobs are launched.  
3. **Use scheduler-specific configuration carefully**: Only add scheduler-specific flags when necessary (e.g., node labels on Kubernetes).  
4. **Test locally first**: Use `local_cwd` or similar local schedulers for rapid iteration before moving to clusters.

## Common Pitfalls

- **Tightly coupling training code to a specific scheduler**: Makes it hard to reuse components across environments.  
- **Assuming file paths/images exist identically on all backends**: Ensure your container images and paths are valid for each scheduler.  
- **Not monitoring jobs after submission**: Use TorchX or scheduler tooling to track job status and logs.


## Performance Optimization

- **Leverage underlying distributed training optimizations** (DDP, FSDP, etc.) within your training script.  
- **Right-size resources per job** (CPUs, GPUs, memory) for the scheduler you’re using.  
- **Use node affinity and placement policies** on schedulers like Kubernetes or Slurm for locality and bandwidth.  
- **Monitor utilization** to adjust job specs (e.g., number of workers, batch sizes).

In [ ]:
# Placeholder for performance benchmarking

print("Benchmark different TorchX scheduler backends with your training workload.")

## Production Deployment

- **Kubernetes**: Use TorchX’s Kubernetes scheduler plugin to submit jobs as K8s resources.  
- **Slurm / HPC**: Use the Slurm scheduler plugin to integrate with job queues.  
- **Ray**: Combine TorchX with Ray clusters to schedule training on Ray.  
- Embed TorchX commands in CI/CD pipelines to automate training and evaluation runs.

## Monitoring and Observability

- Use **scheduler-native monitoring** (Kubernetes, Slurm, Ray dashboards) to view job status and resource utilization.  
- Standardize **logging and metrics** across TorchX-launched jobs for consistent observability.  
- Combine with centralized logging or experiment tracking tools to correlate runs across environments.

## Troubleshooting

- **Job fails immediately**: Check container images, entrypoints, and paths.  
- **Jobs hang in pending state**: Investigate scheduler-level resource constraints or quota issues.  
- **Different behavior between local and cluster runs**: Confirm environment parity (PyTorch/TorchX versions, container images, configs).  
- **Difficulties cleaning up jobs**: Use scheduler-specific commands (e.g., `kubectl delete`, `scancel`) alongside TorchX tooling.

## Comparison with Alternatives

| Aspect | TorchX | Ray Train | Kubeflow Training Operators |
|--------|--------|----------|-----------------------------|
| Focus | Job definition & launcher | Training orchestration & ecosystem | Kubernetes-native training CRDs |
| Schedulers | Local, k8s, Slurm, Ray, etc. | Ray cluster | Kubernetes (TFJob, PyTorchJob, etc.) |
| Framework specificity | PyTorch-centric | Any, but PyTorch-first examples | Framework-specific operators |

Use TorchX when you want a **PyTorch-centric job launcher** that can target multiple schedulers with the same component definitions.

## Resources

- TorchX docs: https://docs.pytorch.org/torchx/latest/  
- Quickstart: https://pytorch.org/torchx/main/quickstart.html  
- Application examples: https://docs.pytorch.org/torchx/latest/examples_apps/index.html

These resources contain real-world examples of training, interpretation, and data preprocessing apps launched with TorchX.